# How do we prepare data for models?
**Topics:** Normalization · Standardization · One-Hot Encoding · Handling Missing Values · Feature Selection · PCA · t-SNE · UMAP


---
## 1. Normalization & Standardization

### What it is
- **Normalization (Min-Max)** → scales features to [0, 1]
- **Standardization (Z-score)** → scales to mean=0, std=1
- Both put features on the same scale so no single feature dominates

### Key intuition
- Without scaling: a feature in thousands overpowers one in decimals
- Gradient descent converges much faster on a balanced loss surface

### When to use

**Normalization:**
- Neural networks, image pixel values
- When you need bounded output [0, 1]

**Standardization:**
- Most ML algorithms (linear regression, SVM, PCA, logistic regression)
- When distribution matters more than range
- More robust to outliers than normalization

### When not to use
- Tree-based models (decision trees, random forests, XGBoost) → scaling not needed


In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler

X = np.array([[1000, 0.1], [2000, 0.5], [3000, 0.9], [500, 0.2]])

# Normalization
minmax = MinMaxScaler()
X_norm = minmax.fit_transform(X)

# Standardization
standard = StandardScaler()
X_std = standard.fit_transform(X)

print("Original:")
print(X)
print("\nNormalized (Min-Max):")
print(X_norm.round(3))
print("\nStandardized (Z-score):")
print(X_std.round(3))

# IMPORTANT: fit on train, transform on val/test
# scaler.fit(X_train)
# X_train_scaled = scaler.transform(X_train)
# X_test_scaled  = scaler.transform(X_test)  # use same scaler


### Interview Q&A

**Normalization vs standardization — which to choose?**
- Standardization is generally safer — works even with outliers
- Normalization is preferred when algorithm expects bounded input (e.g. neural nets, k-NN)

**What happens if you fit the scaler on the full dataset?**
- Data leakage — val/test statistics contaminate training → always fit on train only

### Gotchas
- Apply scaling after train/test split, never before
- Re-use the same fitted scaler for inference on new data


---
## 2. One-Hot Encoding

### What it is
- Converts categorical variables into binary columns (0 or 1)
- One column per category — exactly one column is 1 per row

### Key intuition
- ML models work with numbers, not strings
- Without encoding, "Red=1, Blue=2, Green=3" implies Red < Blue < Green — false ordering
- One-hot removes this false ordering by giving each category its own column

### When to use
- Nominal categories with no natural order (color, city, product type)
- Algorithms that are sensitive to numeric magnitude (linear models, SVM, neural nets)

### When not to use
- High cardinality features (100+ categories) → too many columns, use target encoding instead
- Tree-based models handle categories natively in some implementations
- Ordinal data (low/medium/high) → use ordinal encoding to preserve order


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

df = pd.DataFrame({
    'color'  : ['red', 'blue', 'green', 'red', 'blue'],
    'size'   : ['small', 'large', 'medium', 'large', 'small'],
    'price'  : [10, 25, 15, 30, 12]
})

# One-hot encoding (nominal)
ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop first avoids dummy variable trap
color_encoded = ohe.fit_transform(df[['color']])
print("One-Hot Encoded (color):")
print(pd.DataFrame(color_encoded, columns=ohe.get_feature_names_out()))

# Ordinal encoding (ordered categories)
oe = OrdinalEncoder(categories=[['small', 'medium', 'large']])
size_encoded = oe.fit_transform(df[['size']])
print("\nOrdinal Encoded (size):")
print(size_encoded.ravel())  # small=0, medium=1, large=2

# Pandas get_dummies (quick alternative)
print("\nPandas get_dummies:")
print(pd.get_dummies(df['color'], drop_first=True).astype(int))


### Interview Q&A

**What is the dummy variable trap?**
- With N categories, N-1 columns are enough — the Nth is always derivable
- Having all N columns causes multicollinearity → use `drop='first'`

**When would you use target encoding instead of one-hot?**
- High cardinality features — replace each category with mean of target for that category
- Risk: data leakage if not done within cross-validation folds

### Gotchas
- Unseen categories at inference time → handle with `handle_unknown='ignore'` in sklearn
- Don't one-hot encode the target variable in classification


---
## 3. Handling Missing Values

### What it is
- Strategies to deal with NaN/null values in the dataset
- Missing data can bias models or cause errors if not handled

### Strategies

| Strategy | When to use |
|---|---|
| Drop rows | Very few missing rows, data is abundant |
| Drop columns | Column has >50% missing values |
| Mean/median imputation | Numerical, missing at random |
| Mode imputation | Categorical features |
| Model-based imputation | Complex patterns, features correlated |
| Indicator column | Missingness itself is informative |

### Key intuition
- Never just ignore missing values — most algorithms can't handle NaN
- The pattern of missingness can itself be a signal (e.g. income not reported = likely high earner)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer

np.random.seed(42)
df = pd.DataFrame({
    'age'    : [25, np.nan, 35, 40, np.nan, 28],
    'income' : [50000, 60000, np.nan, 80000, 55000, np.nan],
    'city'   : ['NY', 'LA', np.nan, 'NY', 'LA', 'NY']
})

print("Missing values per column:")
print(df.isnull().sum())
print()

# Mean imputation (numerical)
imp_mean = SimpleImputer(strategy='mean')
df['age_imputed'] = imp_mean.fit_transform(df[['age']])

# Median imputation (more robust to outliers)
imp_median = SimpleImputer(strategy='median')
df['income_imputed'] = imp_median.fit_transform(df[['income']])

# Mode imputation (categorical)
imp_mode = SimpleImputer(strategy='most_frequent')
df['city_imputed'] = imp_mode.fit_transform(df[['city']])

# Add missingness indicator
df['age_missing'] = df['age'].isnull().astype(int)

# KNN imputation (uses neighboring values)
knn_imp = KNNImputer(n_neighbors=2)
df[['age_knn', 'income_knn']] = knn_imp.fit_transform(df[['age', 'income']])

print(df.round(1))


### Interview Q&A

**Mean vs median imputation — when to use which?**
- Mean → when feature is roughly normally distributed
- Median → when feature has outliers (median is more robust)

**What is MCAR, MAR, MNAR?**
- MCAR (Missing Completely at Random) → safe to impute or drop
- MAR (Missing at Random) → missingness depends on other observed features → model-based imputation
- MNAR (Missing Not at Random) → missingness depends on the missing value itself → hardest case, add indicator column

### Gotchas
- Fit imputer on train set only — transform val/test with same imputer
- Imputing target variable is almost always wrong — drop those rows instead


---
## 4. Feature Selection

### What it is
- Choosing which features to include in the model
- Reduces dimensionality, removes noise, speeds up training

### Methods

| Method | How | When |
|---|---|---|
| Filter (correlation, variance) | Statistical score, no model | Quick first pass |
| Wrapper (RFE) | Train model, remove weakest feature, repeat | Smaller datasets |
| Embedded (L1, tree importance) | Built into model training | Most practical choice |

### When to use
- Many features relative to samples → risk of overfitting
- Features are correlated → redundant information
- Need to improve inference speed

### When not to use
- Few features already — selection may remove useful signal
- Deep learning — learns feature importance internally


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.feature_selection import (SelectKBest, f_classif,
                                        RFE, SelectFromModel)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

X, y = make_classification(n_samples=300, n_features=15,
                            n_informative=5, n_redundant=5, random_state=42)

# 1. Filter: Select top K features by ANOVA F-score
selector_f = SelectKBest(f_classif, k=5)
X_filter = selector_f.fit_transform(X, y)
print(f"Filter selected features: {selector_f.get_support().nonzero()[0]}")

# 2. Wrapper: Recursive Feature Elimination (RFE)
rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=5)
rfe.fit(X, y)
print(f"RFE selected features   : {rfe.get_support().nonzero()[0]}")

# 3. Embedded: Feature importance from Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)
importances = rf.feature_importances_
top5 = np.argsort(importances)[::-1][:5]
print(f"RF top 5 features       : {top5}")
print(f"Importances             : {importances[top5].round(3)}")


### Interview Q&A

**Filter vs wrapper vs embedded — which is best?**
- Embedded (L1 / tree importance) is usually the best tradeoff — fast and model-aware
- Wrapper (RFE) is most thorough but slow on large feature sets
- Filter is a good quick first pass to remove obviously useless features

**What is the curse of dimensionality?**
- As features increase, data becomes sparse in high-dimensional space
- Distances become less meaningful — k-NN and clustering suffer most

### Gotchas
- Run feature selection inside cross-validation folds — not before splitting
- High correlation between features ≠ feature is useless (it may still add signal)


---
## 5. PCA (Principal Component Analysis)

### What it is
- Dimensionality reduction — transforms features into new axes (principal components) ordered by variance explained
- Keep top N components → fewer features, minimal information loss

### Key intuition
- Data cloud shaped like a pancake → PCA finds the flat directions
- Principal components = directions of maximum variance in data
- Found via eigenvectors of the covariance matrix

### When to use
- Many correlated features → reduce redundancy
- Visualizing high-dimensional data (reduce to 2–3)
- Speed up downstream model training

### When not to use
- Interpretability matters — components lose original feature meaning
- Features already low-dimensional and uncorrelated
- Need non-linear structure → use t-SNE or UMAP instead

### Connection to File 3
- Always **standardize before PCA** — large-scale features dominate otherwise
- Natural flow: standardize → PCA → model


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ── From scratch ──
def pca_from_scratch(X, n_components):
    X = (X - X.mean(axis=0)) / X.std(axis=0)           # standardize
    cov_matrix = np.cov(X.T)                             # covariance matrix
    eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
    idx = np.argsort(eigenvalues)[::-1]                 # sort descending
    components = eigenvectors[:, idx][:, :n_components]
    return X @ components

# ── scikit-learn ──
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=200, n_features=10, random_state=42)

X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X_scaled)
print(f"Explained variance ratio : {pca.explained_variance_ratio_.round(3)}")
print(f"Total variance kept      : {pca.explained_variance_ratio_.sum():.3f}")

# Choosing n_components — pick where cumulative variance crosses 95%
pca_full = PCA().fit(X_scaled)
cumvar = pca_full.explained_variance_ratio_.cumsum()
n = (cumvar < 0.95).sum() + 1
print(f"Components for 95% var   : {n}")


### Interview Q&A

**PCA vs feature selection?**
- Feature selection → keeps original features, drops some
- PCA → blends all features into new components, none are kept as-is

**Why standardize before PCA?**
- High-scale features dominate variance → standardization levels the playing field

**What does explained variance ratio tell you?**
- Fraction of total variance each component captures
- Pick N where cumulative sum crosses ~95%

**PCA vs t-SNE?**
- PCA → linear, global structure, reusable on new data → use for preprocessing
- t-SNE → non-linear, local clusters, not reusable → use for visualization only

### Gotchas
- Linear only → misses curved structure (use kernel PCA or autoencoders)
- Outliers distort components — clean data first
- Low explained variance (e.g. 40% in 2 components) → PCA losing too much info


---
## 6. t-SNE

### What it is
- Non-linear dimensionality reduction technique for visualization
- Maps high-dimensional data to 2D or 3D while preserving local structure (nearby points stay nearby)

### Key intuition
- Converts distances between points into probabilities
- Minimizes difference between high-dim and low-dim probability distributions
- Clusters in 2D reflect genuine groupings in high-dimensional space

### When to use
- Visualizing clusters in high-dimensional data (images, embeddings, text)
- Exploring whether natural groupings exist

### When not to use
- Preprocessing for a model — output is not reusable on new data
- Interpreting distances between clusters — not meaningful in t-SNE output
- Large datasets (>50k samples) — very slow, use UMAP instead


In [ ]:
from sklearn.manifold import TSNE
from sklearn.datasets import load_digits
import numpy as np

digits = load_digits()
X, y = digits.data, digits.target   # 1797 samples, 64 features (8x8 images)

tsne = TSNE(
    n_components=2,
    perplexity=30,      # balance local vs global structure (try 5–50)
    random_state=42,
    max_iter=1000
)
X_tsne = tsne.fit_transform(X)

print(f"Input shape  : {X.shape}")
print(f"Output shape : {X_tsne.shape}")
print("t-SNE complete — plot X_tsne colored by y to see digit clusters")


### Interview Q&A

**What is perplexity in t-SNE?**
- Roughly controls the number of nearest neighbors considered
- Low perplexity → focus on very local structure
- High perplexity → more global structure preserved
- Typical range: 5–50, default 30

**Why can't you use t-SNE for new data points?**
- t-SNE has no parametric mapping — it optimizes positions for the specific dataset
- For reusable embeddings use UMAP or an autoencoder

### Gotchas
- Different random seeds produce different layouts — always set random_state
- Cluster sizes and distances in t-SNE output are not interpretable
- Very slow on large datasets → subsample or use UMAP


---
## 7. UMAP

### What it is
- Non-linear dimensionality reduction — faster alternative to t-SNE
- Preserves both local and global structure better than t-SNE
- Can be used for both visualization and as a preprocessing step (unlike t-SNE)

### Key intuition
- Constructs a graph of nearest neighbors in high-dim space
- Optimizes a low-dim layout that preserves this graph structure
- Based on Riemannian geometry and algebraic topology (but you don't need the math)

### When to use
- Visualization of high-dimensional data (clusters, embeddings)
- Large datasets — much faster than t-SNE
- When you need to transform new data points (unlike t-SNE)

### When not to use
- Need strict mathematical guarantees on distance preservation
- Hyperparameter sensitivity is a concern and you can't tune carefully


In [ ]:
# pip install umap-learn
import numpy as np
from sklearn.datasets import load_digits

try:
    import umap

    digits = load_digits()
    X, y = digits.data, digits.target

    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=15,    # controls local vs global structure (like perplexity in t-SNE)
        min_dist=0.1,      # controls how tightly points cluster (0.0 to 1.0)
        random_state=42
    )
    X_umap = reducer.fit_transform(X)

    print(f"Input shape  : {X.shape}")
    print(f"Output shape : {X_umap.shape}")

    # UMAP can transform new data (unlike t-SNE)
    X_new = X[:5]
    X_new_transformed = reducer.transform(X_new)
    print(f"New data transformed: {X_new_transformed.shape}")

except ImportError:
    print("Install umap-learn: pip install umap-learn")
    print("Usage: umap.UMAP(n_neighbors=15, min_dist=0.1).fit_transform(X)")


### Interview Q&A

**UMAP vs t-SNE?**
- UMAP is faster, scales to larger datasets
- UMAP preserves more global structure
- UMAP can transform new data points — t-SNE cannot
- Both are primarily for visualization

**What do n_neighbors and min_dist control?**
- n_neighbors → low = local structure, high = global structure (like perplexity)
- min_dist → low = tight clusters, high = spread out layout

### Gotchas
- Results still depend on hyperparameters — always try a few settings
- Not fully deterministic even with random_state set across different library versions
- Global distances between clusters more meaningful than t-SNE but still not perfectly interpretable
